In [1]:
import pandas as pd 
import numpy as np 

In [2]:
data = pd.read_csv('U.S._Chronic_Disease_Indicators__CDI___2023_Release copy.csv', low_memory=False)

In [ ]:
data.shape

In [4]:
df1 = pd.read_csv('MEDICAID_AGGREGATE20.CSV')

In [ ]:
conda install openpyxl

In [ ]:
data.columns

In [ ]:
data['DataValueType'].value_counts()

In [8]:
#select relevant columns only and rename them 
filtered_df = (
    data[
        (data['DataValueType'] == 'Number') & 
        (data['StratificationCategory1'].isin(['Gender', 'Race/Ethnicity']))
    ][['YearStart', 'LocationDesc', 'Topic', 'DataValue', 'StratificationCategory1', 'Stratification1']]
    .rename(columns={
        'YearStart': 'Year',
        'LocationDesc': 'State',
        'Topic': 'Disease',
        'DataValue': 'Number_Diagnosed'
    })
    .assign(
        Year=lambda df: pd.to_numeric(df['Year'], errors='coerce'),
        Number_Diagnosed=lambda df: pd.to_numeric(df['Number_Diagnosed'], errors='coerce')
    )
    .sort_values(by=['Year', 'State'])
)

In [9]:
data = filtered_df.copy()

In [10]:
data = data[data['Disease']=='Cardiovascular Disease']

In [ ]:
data.isnull().sum()

In [12]:
#remove rows with na in 'number_diagnosed'
data= data.dropna()

In [ ]:
data.isnull().sum()

In [14]:
df1 = pd.read_csv('MEDICAID_AGGREGATE20.CSV')

In [15]:
df1=df1[['State_Name', 'Y2010','Y2011','Y2012','Y2013','Y2014','Y2015','Y2016','Y2017','Y2018','Y2019','Y2020']]

In [16]:
medicaid_df = df1.dropna()


In [ ]:
medicaid_df.head()

In [ ]:
# reshape medicaid df to long format and group by state then sum all medicaid expenses
# 1. Melt the DataFrame (pivot_longer equivalent)
medicaid_long = pd.melt(medicaid_df, 
                        id_vars=['State_Name'], 
                        value_vars=[col for col in medicaid_df.columns if col.startswith('Y20')],
                        var_name='Year', 
                        value_name='Expenses')

# 2. Clean the 'Year' column by removing the 'Y' and converting to int
medicaid_long['Year'] = medicaid_long['Year'].str.replace('Y', '', regex=False).astype(int)

# 3. Group by State and Year, then sum Expenses
medicaid_grouped = medicaid_long.groupby(['State_Name', 'Year'], as_index=False).agg(
    Medicaid_Expenses=('Expenses', 'sum')
)

print(medicaid_grouped.head())

In [ ]:
combined_df = pd.merge(medicaid_grouped, data, how='left', left_on=['State_Name', 'Year'], right_on=['State', 'Year'])

# Display the first few rows of the combined DataFrame
print(combined_df.head())

In [20]:
# select columns with data from 2010 through 2020
df2 = pd.read_excel('h08.xlsx', engine='openpyxl')

In [ ]:
df2 = df2.iloc[6:, :]
df2.head()

In [ ]:
df2.columns = df2.iloc[0, :]
df2 = df2.iloc[1:, :]
df2.head()

In [23]:
df2= df2[['State', '2020 (41)', 2019,2018,'2017 (40)',2016, 2015,2014,'2013 (39)',2012,2011,'2010 (37)']]

In [24]:
df2.rename(columns ={'2020 (41)':'2020', '2017 (40)':'2017','2013 (39)':'2013','2010 (37)':'2010'}, inplace=True)

In [ ]:
df2 = df2.drop(index=7)
df2.head()

In [26]:
df2=df2[~df2['State'].isin(['United States','District of Columbia','2023 Dollars','State'])]

In [27]:
df2.columns = [str(col) if isinstance(col, int) or (isinstance(col, str) and col.isdigit()) else col for col in df2.columns]


In [28]:
long_df2 = df2.melt(
    id_vars=[col for col in df2.columns if col not in [str(year) for year in range(2010, 2021)]],
    value_vars=[str(year) for year in range(2010, 2021)],
    var_name='Year',
    value_name='Median_Income'
)

In [ ]:
long_df2.head()

In [ ]:
long_df2['Year'] = long_df2['Year'].astype(int)
long_df2.info()

In [ ]:
final_df = pd.merge(combined_df, long_df2, how='left', left_on=['State_Name', 'Year'], right_on=['State', 'Year'])

# Display the first few rows of the combined DataFrame
print(final_df.head())

In [32]:
final_df.drop(['State_x', 'State_y'], axis = 1,inplace=True)

In [33]:
final_df = final_df.dropna()

In [ ]:
final_df.isnull().sum()

In [35]:
final_df['Median_Income'] = pd.to_numeric(final_df['Median_Income'], errors='coerce')

In [ ]:
final_df.info()

# Exploratory data analysis 

In [ ]:
# summary stats for numeric columns
final_df.describe()

In [ ]:
final_df.groupby('Stratification1', as_index=False)['Number_Diagnosed'].mean()

In [ ]:
# summary stats by state
final_df.groupby('State_Name')[['Medicaid_Expenses', 'Number_Diagnosed', 'Median_Income']].agg(['mean', 'median'])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style for seaborn (optional)
sns.set(style="whitegrid")

# Create the plot
plt.figure(figsize=(8, 6))
sns.histplot(final_df['Medicaid_Expenses'], bins=range(0, int(final_df['Medicaid_Expenses'].max()) + 10000, 10000), 
             color="cadetblue", kde=False, edgecolor="black")

# Adding titles and labels
plt.title("Histogram of Medicaid Expenses")
plt.xlabel("Medicaid Expenses (in millions)")
plt.ylabel("Frequency")

# Show the plot
plt.show()

In [ ]:
# Set the style for seaborn (optional)
sns.set(style="whitegrid")

# Create the plot
plt.figure(figsize=(8, 6))
sns.histplot(final_df['Median_Income'], bins=range(0, int(final_df['Median_Income'].max()) + 1000, 1000), 
             color="cadetblue", kde=False, edgecolor="black")

# Adding titles and labels
plt.title("Histogram of Median Income")
plt.xlabel("Median Income (in thousands)")
plt.ylabel("Frequency")

# Show the plot
plt.show()

In [ ]:
# Filter to gender
gender = final_df[final_df['StratificationCategory1'] == 'Gender']

# Filter to race/ethnicity
race = final_df[final_df['StratificationCategory1'] == 'Race/Ethnicity']

# Calculate the IQR (Interquartile Range) for both datasets
Q1_gender = gender['Number_Diagnosed'].quantile(0.25)
Q3_gender = gender['Number_Diagnosed'].quantile(0.75)
IQR_gender = Q3_gender - Q1_gender

Q1_race = race['Number_Diagnosed'].quantile(0.25)
Q3_race = race['Number_Diagnosed'].quantile(0.75)
IQR_race = Q3_race - Q1_race

# Set the style for seaborn (optional)
sns.set(style="whitegrid")

# Boxplot by gender with zoom on IQR (whis=1.5 by default, but we're explicitly focusing on IQR)
plt.figure(figsize=(8, 6))
sns.boxplot(data=gender, x='Stratification1', y='Number_Diagnosed', color="orange", whis=1.5)
plt.title("Boxplot by Gender")
plt.xlabel("Gender")
plt.ylabel("Number Diagnosed")
plt.xticks(rotation=45)  # Optional: Rotate labels if they are long
plt.ylim(Q1_gender - 1.5 * IQR_gender, Q3_gender + 1.5 * IQR_gender)  # Set y-axis limits based on IQR
plt.show()

# Boxplot by race/ethnicity with zoom on IQR
plt.figure(figsize=(8, 6))
sns.boxplot(data=race, x='Stratification1', y='Number_Diagnosed', color="orange", whis=1.5)
plt.title("Boxplot by Race/Ethnicity")
plt.xlabel("Race/Ethnicity")
plt.ylabel("Number Diagnosed")
plt.xticks(rotation=45)  # Optional: Rotate labels if they are long
plt.ylim(Q1_race - 1.5 * IQR_race, Q3_race + 1.5 * IQR_race)  # Set y-axis limits based on IQR
plt.show()

In [ ]:
# Scatter plot showing number diagnosed against median income
plt.figure(figsize=(8, 6))
sns.scatterplot(data=final_df, x='Median_Income', y='Number_Diagnosed', color='red')
plt.title("Scatter Plot for Number Diagnosed vs. Median Income")
plt.xlabel("Median Income (in thousands)")
plt.ylabel("Number Diagnosed")
plt.show()

In [ ]:
# Scatter plot showing number diagnosed against Medicaid expenses
plt.figure(figsize=(8, 6))
sns.scatterplot(data=final_df, x='Medicaid_Expenses', y='Number_Diagnosed', color='red')
plt.title("Scatter Plot for Number Diagnosed vs. Medicaid Expenses")
plt.xlabel("Medicaid Expenses (in millions)")
plt.ylabel("Number Diagnosed")
plt.show()

In [ ]:
# Select only numeric columns
numeric_df = final_df.select_dtypes(include=['number'])

# Calculate the correlation matrix
cor_matrix = numeric_df.corr()

# Plot the correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cor_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
X = final_df.drop(['Number_Diagnosed', 'Disease', 'StratificationCategory1'], axis=1)
X.info()

# Modeling

In [47]:
from sklearn.model_selection import train_test_split

# Set the seed for reproducibility
import numpy as np
np.random.seed(123)

# Assuming modified_df is your DataFrame
X = final_df.drop(['Number_Diagnosed', 'Disease', 'StratificationCategory1'], axis=1)  # Features (excluding the target)
X = pd.get_dummies(X, columns=['Stratification1', 'State_Name'], drop_first=False)
train_X = X[(X['Year'] >= 2010) & (X['Year'] <= 2017)]
test_X = X[(X['Year'] >= 2018) & (X['Year'] <= 2020)]
train_y = final_df[(final_df['Year'] >= 2010) & (final_df['Year'] <= 2017)]['Number_Diagnosed']  # Target variable
test_y = final_df[(final_df['Year'] >= 2018) & (final_df['Year'] <= 2020)]['Number_Diagnosed']

In [ ]:
!pip install xgboost

In [49]:
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge


In [ ]:
# Linear Regression
linear_model = LinearRegression()
linear_model.fit(train_X, train_y)

# Ridge Regression (with alpha=1.0 as an example, you can tune this)
ridge_model = Ridge(alpha=1.0, random_state=42)
ridge_model.fit(train_X, train_y)

# Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(train_X, train_y)

# Gradient Boosting Regressor
gbm_model = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.01, random_state=42)
gbm_model.fit(train_X, train_y)

# XGBoost Regressor
xgb_model = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.01, random_state=42)
xgb_model.fit(train_X, train_y)

# Decision Tree Regressor
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(train_X, train_y)

# Support Vector Regressor (SVM)
svm_model = SVR()
svm_model.fit(train_X, train_y)

# K-Nearest Neighbors Regressor
knn_model = KNeighborsRegressor(n_neighbors=5)
knn_model.fit(train_X, train_y)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Make predictions on the test set
# Predict on test set
linear_pred = linear_model.predict(test_X)
ridge_pred = ridge_model.predict(test_X)
rf_pred = rf_model.predict(test_X)
gbm_pred = gbm_model.predict(test_X)
xgb_pred = xgb_model.predict(test_X)
dt_pred = dt_model.predict(test_X)
svm_pred = svm_model.predict(test_X)
knn_pred = knn_model.predict(test_X)

# Define a function to compute evaluation metrics
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"\n{model_name} Performance:")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R² Score: {r2:.4f}")

# Evaluate each model
evaluate_model(test_y, linear_pred, "Linear Regression")
evaluate_model(test_y, ridge_pred, "Ridge Regression")
evaluate_model(test_y, rf_pred, "Random Forest Regressor")
evaluate_model(test_y, gbm_pred, "Gradient Boosting Regressor")
evaluate_model(test_y, xgb_pred, "XGBoost Regressor")
evaluate_model(test_y, dt_pred, "Decision Tree Regressor")
evaluate_model(test_y, svm_pred, "Support Vector Regressor")
evaluate_model(test_y, knn_pred, "K-Nearest Neighbors Regressor")

Random Forest outperforms others in almost every metric. It explains ~73.5% of the variance (R²) and produces the smallest errors.

# Forecasting (2021-2050)

Step 1: Prepare the Data and Feature Selection

In [61]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Assuming final_df is your DataFrame

# Features (exclude target columns)
X = final_df.drop(['Number_Diagnosed', 'Disease', 'StratificationCategory1'], axis=1)

# One-hot encoding for categorical columns
X = pd.get_dummies(X, columns=['Stratification1', 'State_Name'], drop_first=False)

# Prepare the target variables for prediction
# Medicaid_Expenses and Median_Income
y_medicaid = final_df['Medicaid_Expenses']
y_income = final_df['Median_Income']

# Train-test split (we'll use the years from 2010 to 2020 for training, and 2021-2050 for prediction)
train_X = X[X['Year'] <= 2020]
test_X = X[X['Year'] > 2020]
train_y_medicaid = y_medicaid[X['Year'] <= 2020]
train_y_income = y_income[X['Year'] <= 2020]

Step 2: Train the Random Forest Models

In [ ]:
# Initialize the Random Forest Regressors for Medicaid and Median Income
rf_model_medicaid = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_income = RandomForestRegressor(n_estimators=100, random_state=42)

# Train the models
rf_model_medicaid.fit(train_X, train_y_medicaid)
rf_model_income.fit(train_X, train_y_income)

Step 3: Prepare Future Data for Prediction (2021-2050)

In [ ]:
# Generate the years, states, and stratification values
years = np.arange(2021, 2051)
states = final_df['State_Name'].unique()
stratifications = final_df['Stratification1'].unique()

# Create the future DataFrame for predictions
future_df = pd.DataFrame([(state, year, strat)
                           for state in states
                           for year in years
                           for strat in stratifications],
                          columns=['State_Name', 'Year', 'Stratification1'])

# Add placeholder values for features (like Medicaid_Expenses and Median_Income)
future_df['Medicaid_Expenses'] = final_df['Medicaid_Expenses'].mean()
future_df['Median_Income'] = final_df['Median_Income'].mean()

# One-hot encoding for the categorical columns
future_X = pd.get_dummies(future_df, columns=['Stratification1', 'State_Name'], drop_first=False)

# View the first few rows of future data
print(future_df.head())

Step 4: Predict Future Values for Medicaid and Median Income

In [ ]:
# Predict Medicaid_Expenses and Median_Income for the future years (2021-2050)
future_df['Predicted_Medicaid_Expenses'] = rf_model_medicaid.predict(future_X)
future_df['Predicted_Median_Income'] = rf_model_income.predict(future_X)

# View the predictions for the first few rows
print(future_df.head())

Step 5 : Save the Predictions to an Excel File

In [ ]:
# Save the predictions to an Excel file
future_df.to_excel("predictions_2021_2050_with_state_random_forest.xlsx", index=False, sheet_name="Predictions")

# Output a message
print("Predictions saved to 'predictions_2021_2050_with_state_random_forest.xlsx'")